## Action for Anando Prokumar

This notebook and environment changes are intended to be applied by: **Anando Prokumar**.

Checklist:
- Run the environment update in a terminal: `conda env update -f environment.yaml --prune -n vision-agent`
- Register kernel as described in the notebook
- Launch Jupyter Lab or Notebook and select the `vision-agent` kernel
- Run the cells in this notebook and then run `python -m src.main` or the cell that runs the pipeline

(If you prefer manual steps, see the 'Setup' section below.)


# HYGO: Environment and Pipeline Runner

This notebook automates the environment update and provides steps to run the HYGO pipeline or a small Pi estimation script.

Intended for: Anando Prokumar — please update the environment using the cells below, register the kernel, and run the pipeline.

> Note: The notebook shows commands that may require administrator or interactive steps (e.g., conda activation). Confirm your shell and environment before running.


In [ ]:
# 1) Read and validate environment.yaml

import pathlib

env_path = pathlib.Path(r"c:\Users\maste\OneDrive\Desktop\All-Desktop-Files\coding\interview\HYGO\environment.yaml")
print("Path exists:", env_path.exists())
print("\n--- environment.yaml contents (first 2000 characters) ---\n")
print(env_path.read_text()[:2000])


In [ ]:
# 2) Backup the original environment.yaml

import shutil
backup = env_path.with_suffix('.yaml.bak')
shutil.copy(env_path, backup)
print(f"Backup created: {backup} -> Exists: {backup.exists()}")


In [ ]:
# 3) Add Notebook, IPython kernel, and JupyterLab (progammatically update env file)

# Install pyyaml locally in this kernel so we can edit the YAML file
!pip install -q pyyaml

import yaml
import pathlib

data = yaml.safe_load(env_path.read_text())

def find_pip_list(data):
    for item in data.get('dependencies', []):
        if isinstance(item, dict) and 'pip' in item:
            return item['pip']
    # If pip not found, append a new pip list and return it
    pip_item = {'pip': []}
    data['dependencies'].append(pip_item)
    return pip_item['pip']

pip_list = find_pip_list(data)
print('Existing pip packages:', len(pip_list))

# packages to ensure
needed = ['notebook', 'ipykernel', 'jupyterlab']
for pkg in needed:
    if pkg not in pip_list:
        pip_list.append(pkg)

# Write back the YAML (keep a simple ordering)
env_path.write_text(yaml.safe_dump(data, sort_keys=False))
print('\nUpdated environment.yaml updated. Review changes above.')


In [ ]:
# 4) Update the conda environment using the modified environment.yaml

import subprocess
import shlex

cmd = f'conda env update -f "{env_path}" --prune'
print('Running:', cmd)

proc = subprocess.run(shlex.split(cmd), capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
if proc.returncode != 0:
    print('Warning: conda env update returned non-zero status. Check output above.')
else:
    print('Environment update completed (check output).')


In [ ]:
# 5) Register the kernel for the 'vision-agent' environment

import subprocess
cmd = ['conda', 'run', '-n', 'vision-agent', 'python', '-m', 'ipykernel', 'install', '--user', '--name', 'vision-agent', '--display-name', 'vision-agent']
print('Running kernel install:', ' '.join(cmd))
try:
    subprocess.run(cmd, check=True)
    subprocess.run(['jupyter', 'kernelspec', 'list'], check=True)
except subprocess.CalledProcessError as e:
    print('Kernel register failed:', e)
    print('If this fails, run the following command in your shell:')
    print('\nconda run -n vision-agent python -m ipykernel install --user --name vision-agent --display-name "vision-agent"\n')


In [ ]:
# 6) Create a Python script `main_pi.py` that estimates π (Pi) using Monte Carlo method

import pathlib

script_path = pathlib.Path('main_pi.py')
script_content = r"""import random

def estimate_pi(n=100000):
    inside = 0
    for _ in range(n):
        x = random.random()
        y = random.random()
        if x*x + y*y <= 1:
            inside += 1
    return 4 * inside / n

if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--n', type=int, default=100000)
    args = parser.parse_args()
    print(estimate_pi(args.n))
"""

script_path.write_text(script_content)
print(f"Created {script_path} (size: {script_path.stat().st_size} bytes)")


In [ ]:
# 7) Run the script inside the updated conda environment

import subprocess

cmd = ['conda','run','-n','vision-agent','python','main_pi.py','--n','500000']
print('Running:', ' '.join(cmd))
try:
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print('stdout:\n', proc.stdout)
    print('stderr:\n', proc.stderr)
    if proc.returncode != 0:
        print('Command returned non-zero status code:', proc.returncode)
except Exception as e:
    print('Execution failed:', e)


In [ ]:
# 8) Run the script from the notebook kernel (import and call the function)

import importlib
import main_pi
importlib.reload(main_pi)
print('estimate_pi(100000) =', main_pi.estimate_pi(100000))


In [ ]:
# 9) Create small pytest and run tests for Pi estimate

import pathlib

test_path = pathlib.Path('test_pi.py')

test_content = r"""import main_pi


def test_estimate_pi_accuracy():
    val = main_pi.estimate_pi(50000)
    # Allow a loose tolerance since Monte Carlo is random
    assert abs(val - 3.14159) < 0.08
"""

test_path.write_text(test_content)
print('Created test file:', test_path)

# Run pytest inside environment
import subprocess
cmd = ['conda','run','-n','vision-agent','pytest','-q']
print('Running:', ' '.join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
print('Return code:', proc.returncode)


In [ ]:
# 10) Git commit and push modified files (safe-run)

import subprocess
import shlex

# Add endpoints: This will only run if git remote is configured and you have push access
files_to_commit = ['environment.yaml', 'main_pi.py', 'notebooks/run_pipeline.ipynb']
cmd_add = ['git','add'] + files_to_commit
print('Running:', ' '.join(cmd_add))
proc = subprocess.run(cmd_add, capture_output=True, text=True)
print('git add returncode:', proc.returncode)

commit_msg = 'Add notebook/kernel support and main_pi script; update environment.yaml'
cmd_commit = ['git','commit','-m', commit_msg]
print('Running:', ' '.join(cmd_commit))
proc_commit = subprocess.run(cmd_commit, capture_output=True, text=True)
print(proc_commit.stdout)
print(proc_commit.stderr)

# Push only if commit succeeded
if proc_commit.returncode == 0:
    cmd_push = ['git','push']
    print('Running:', ' '.join(cmd_push))
    proc_push = subprocess.run(cmd_push, capture_output=True, text=True)
    print(proc_push.stdout)
    print(proc_push.stderr)
else:
    print('Nothing committed (maybe no changes). Skipping push.')

print('\nChecklist for Anando Prokumar:')
print('1) Close any terminals and run the commands below in a PowerShell terminal:')
print('   conda env update -f environment.yaml --prune')
print('   conda run -n vision-agent python -m ipykernel install --user --name vision-agent --display-name "vision-agent"')
print('2) Launch Jupyter Lab: jupyter lab')
print("3) Open this notebook and use the 'vision-agent' kernel; then run the cells")


In [ ]:
# Run the HYGO pipeline using the conda environment

import subprocess
cmd = ['conda', 'run', '-n', 'vision-agent', 'python', '-m', 'src.main']
print('Running:', ' '.join(cmd))
proc = subprocess.run(cmd, capture_output=True, text=True)
print('stdout:\n', proc.stdout)
print('stderr:\n', proc.stderr)
print('Return code:', proc.returncode)
